In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from threadpoolctl import threadpool_limits

from study.data import cohort, features, prediction_matrix, prediction_metadata
from study.io import WORK, load, reset_directory, save, write_table
from study.models import apply_calibration, fit_model, fit_recalibrator, predict
from study.observations import feature_view
from study.statistics import paired_difference, prediction_rows

In [ ]:
RULES = {'protected_variables': ['Height', 'MechVent'], 'terminal_outage_hours': 12}
reset_directory(WORK / 'predictions/local_calibration')

source_fits = load(WORK / 'models/source.pkl')
metadata, x = cohort('eicu').sort_index(), features('eicu')
shared = load(WORK / 'feature_sets.pkl')['shared']
local_folds = load(WORK / 'models/local.pkl')

In [ ]:
# Separate fitting, calibration, and test hospitals within each local fold
fitted_folds = {}
for repeat, splits in local_folds.items():
    fitted_folds[repeat] = []

    for fold, split in enumerate(splits):
        # Reserve calibration hospitals within the fitting fold
        train, test = (split['fit'], split['test'])
        fit_position, cal_position = next(
            GroupShuffleSplit(1, test_size = 0.2, random_state = 5 * repeat + fold).split(
                train, groups = metadata.loc[train, 'hospital_id']
            )
        )
        fit, calibration = (train[fit_position], train[cal_position])

        # Fit R2 and R3 without using calibration or test hospitals
        models = {}
        for tier in ('R2', 'R3'):
            with threadpool_limits(limits = 1):
                models[tier] = fit_model(
                    x.loc[fit, shared[tier]],
                    metadata.loc[fit, 'in_hospital_mortality'],
                    'xgboost',
                    source_fits['parameters'],
                )

        fitted_folds[repeat].append({'fit': fit, 'calibration': calibration, 'test': test, 'models': models})

save(WORK / 'models/local_calibration.pkl', fitted_folds)

In [ ]:
# Fit corrections on reserved hospitals under clean or matched inputs
corrections = {}
for calibration_input in ('full', 'retain_0.25', 'terminal_outage'):
    evaluated = x if calibration_input == 'full' else feature_view('eicu', x, calibration_input, 77, RULES)

    for repeat, splits in fitted_folds.items():
        for fold, split in enumerate(splits):
            ids = split['calibration']

            # Score patients from reserved calibration hospitals
            for tier, model in split['models'].items():
                risk = predict(model, evaluated.loc[ids])

                # Fit intercept and Platt corrections on calibration risks
                for method in ('intercept', 'platt'):
                    correction = fit_recalibrator(metadata.loc[ids, 'in_hospital_mortality'], risk, method)
                    corrections[repeat, fold, tier, calibration_input, method] = correction

In [ ]:
# Share evaluation draws across fitted models and correction methods
rows = []
for condition in ('full', 'retain_0.25', 'terminal_outage'):
    seeds = list(range(10)) if condition.startswith('retain_') else [77]

    # Rebuild evaluation inputs once for each observation draw
    for draw, seed in enumerate(seeds):
        evaluated = x if condition == 'full' else feature_view('eicu', x, condition, seed, RULES)

        for repeat, splits in fitted_folds.items():
            rows = []

            for fold, split in enumerate(splits):
                # Score patients from hospitals excluded from fitting and calibration
                ids = split['test']

                for tier, model in split['models'].items():
                    raw = predict(model, evaluated.loc[ids])
                    rows.append(
                        prediction_rows(
                            prediction_metadata(metadata.loc[ids]),
                            raw,
                            tier = tier,
                            fold = fold,
                            repeat = repeat,
                            restriction = condition,
                            calibration = 'raw',
                        ).assign(draw = draw)
                    )

                    # Apply each calibration correction to the same raw risks
                    for calibration_input in ('full', 'retain_0.25', 'terminal_outage'):
                        for method in ('intercept', 'platt'):
                            risk = apply_calibration(
                                corrections[repeat, fold, tier, calibration_input, method], raw
                            )

                            rows.append(
                                prediction_rows(
                                    prediction_metadata(metadata.loc[ids]),
                                    risk,
                                    tier = tier,
                                    fold = fold,
                                    repeat = repeat,
                                    restriction = condition,
                                    calibration = calibration_input + ':' + method,
                                ).assign(draw = draw)
                            )

            write_table(
                WORK / 'predictions/local_calibration' / f'{condition}_{draw}_{repeat}.parquet',
                pd.concat(rows, ignore_index = True),
            )

In [ ]:
# Compare raw and matched calibration process increments
rows = []
for condition in ('full', 'retain_0.25', 'terminal_outage'):
    for calibration in ('raw', condition + ':platt'):
        # Align R3 and R2 risks on the same evaluation patients
        filters = [('restriction', '==', condition), ('calibration', '==', calibration)]
        left = prediction_matrix('local_calibration', metadata.index, filters + [('tier', '==', 'R3')])
        right = prediction_matrix('local_calibration', metadata.index, filters + [('tier', '==', 'R2')])

        for equal_hospital in (False, True):
            result = paired_difference(
                prediction_metadata(metadata), left, right, cluster = True, equal_hospital = equal_hospital
            )
            rows.append(dict(restriction = condition, calibration = calibration, **result))

write_table(WORK / 'results/local_calibration.parquet', pd.DataFrame(rows))